If you've ever tried to force an LLM to output a specific JSON format. It usually throws in conversational filler like "Sure, here is your JSON!" or misses a comma, completely breaking your code when you try to parse it.

local models are especially prone to this. To build reliable software, we need guaranteed data structures, not flaky strings.

To solve this, we will use two heavy-hitting Python libraries:
* Pydantic: The industry standard for defining data schemas in Python.
* Instructor: A brilliant library that acts as a middleman. It takes your Pydantic schema, translates it into strict instructions for the LLM, and automatically validates the output. If the model messes up, Instructor actually re-prompts the model behind the scenes to fix its own error!

In [1]:
pip install instructor pydantic openai

  Using cached instructor-1.14.5-py3-none-any.whl.metadata (12 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached openai-2.29.0-py3-none-any.whl.metadata (29 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  U

In [3]:
import instructor
from openai import OpenAI
from pydantic import BaseModel

# 1. Define exactly what data we want using Pydantic
    # Defining the Blueprint (Pydantic): create a strict "blueprint." 
    # We are telling Python: "I expect a data object called UserProfile. 
    # It MUST contain a name (text), an age (a whole number), and an is_student flag (True or False).
# BaseModel inheits all the basic definitions"Hey, make a new class called UserProfile, and let it inherit all the superpowers of Pydantic's BaseModel."
# 1. Strict Type Validation: Because of BaseModel, your code actively guards your data. 
# 2. Smart Parsing (Type Coercion): If the LLM outputs {"age": "22"} (a string containing a number),  it automatically converts the string "22" into the integer 22 for you.
# 3. Because it's a BaseModel, it comes with built-in methods like .model_dump_json()
class UserProfile(BaseModel):
    name: str
    age: int
    is_student: bool

# 2. Set up the Instructor client to point to our local Ollama engine
    # Ollama is designed to mimic the standard OpenAI API. By pointing the standard OpenAI Python client to our local Ollama address (localhost:11434/v1), 
    # we can use enterprise-grade tools built for OpenAI with our free, local 8B model.
# We wrap that client in instructor.from_openai(). Instructor acts like a bouncer at a club. It intercepts our requests and ensures the model knows it must output JSON.
client = instructor.from_openai(
    OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama", # The SDK requires a key, but Ollama ignores it
    ),
    mode=instructor.Mode.JSON,
)

# 3. Pass raw text and force it into the UserProfile schema
    # When we pass response_model=UserProfile, Instructor does something incredibly clever. 
    # Before sending the request to Llama 3.1, it quietly rewrites your prompt behind the scenes. 
    # It injects the UserProfile schema instructions so the model knows exactly how to format its answer.
response = client.chat.completions.create(
    model="llama3.1",
    response_model=UserProfile,
    messages=[
        {
            "role": "user", 
            "content": "My name is Sarah, I just celebrated my 22nd birthday yesterday, and I'm currently taking a heavy course load at the local university."
        }
    ]
)

# 4. Print the perfectly structured Python object!
print("Type:", type(response))
print("Data:", response.model_dump_json(indent=2))

Type: <class '__main__.UserProfile'>
Data: {
  "name": "Sarah",
  "age": 22,
  "is_student": true
}
